In [ ]:
import pandas as pd
from itertools import permutations

In [ ]:
# # EXO
protein_complex =['SKIV2L2', 'ZCCHC8', 'EXOSC5', 'EXOSC3', 'RBM7', 'EXOSC6', 'EXOSC9', 'EXOSC10', 'C1D', 'EXOSC2', 'EXOSC1', 'EXOSC4', 'EXOSC7', 'MPHOSPH6', 'DIS3L']

# # FANC
# protein_complex =["FANCB", "FANCC", "FANCE"]

# # THO
# protein_complex =["THOC1", "THOC2", "THOC3", "THOC5", "THOC6", "THOC7"]

# # NSL
# protein_complex = ["DUS2", "KANSL1", "KANSL2", "MCRS1", "PHF20", "PHF20L1"]

In [ ]:
complex_pairs = list(permutations(protein_complex, 2))
complex_pairs = [pair for pair in complex_pairs if pair[0] < pair[1]]
pairs = [pair[0]+"__"+pair[1] for pair in complex_pairs]

In [ ]:
bioplex_HCT116 = pd.read_csv("../../result/HEK293T_HCT116/ppi/FromRaw_BioPlex_HCT116.ppi", sep="\t", header=None)
bioplex_HEK293 = pd.read_csv("../../result/HEK293T_HCT116/ppi/FromRaw_BioPlex_HEK293T.ppi", sep="\t", header=None)
sec_HCT116 = pd.read_csv("../../result/HEK293T_HCT116/ppi/FromRaw_SECMX_HCT116.ppi", sep="\t", header=None)
sec_HEK293 = pd.read_csv("../../result/HEK293T_HCT116/ppi/FromRaw_SECMX_HEK293T.ppi", sep="\t", header=None)

bioplex_HCT116 = pd.DataFrame(pairs).merge(bioplex_HCT116, left_on=0, right_on=0, how='left')
bioplex_HEK293 = pd.DataFrame(pairs).merge(bioplex_HEK293, left_on=0, right_on=0, how='left')
sec_HCT116 = pd.DataFrame(pairs).merge(sec_HCT116, left_on=0, right_on=0, how='left')
sec_HEK293 = pd.DataFrame(pairs).merge(sec_HEK293, left_on=0, right_on=0, how='left')

In [ ]:
dfs = [bioplex_HCT116, bioplex_HEK293, sec_HCT116, sec_HEK293]

for df in dfs:
    df[1] = (~df[1].isnull()).astype(int)

renamed_dfs = []
for i, df in enumerate(dfs, start=1):
    temp = df.copy()
    temp.columns = ['col0', f'col{i}']  # rename second column
    renamed_dfs.append(temp)

merged_df = renamed_dfs[0]
for temp in renamed_dfs[1:]:
    merged_df = merged_df.merge(temp, on='col0')

In [ ]:
merged_df[[0, 1]] = merged_df['col0'].str.split("__", expand=True)

merged_df['AP'] = "None"
merged_df.loc[(merged_df['col1'] == 1) & (merged_df['col2'] == 1), 'AP'] = "Share"
merged_df.loc[(merged_df['col1'] == 1) & (merged_df['col2'] == 0), 'AP'] = "HCT"
merged_df.loc[(merged_df['col1'] == 0) & (merged_df['col2'] == 1), 'AP'] = "HEK"

merged_df['SEC'] = "None"
merged_df.loc[(merged_df['col3'] == 1) & (merged_df['col4'] == 1), 'SEC'] = "Share"
merged_df.loc[(merged_df['col3'] == 1) & (merged_df['col4'] == 0), 'SEC'] = "HCT"
merged_df.loc[(merged_df['col3'] == 0) & (merged_df['col4'] == 1), 'SEC'] = "HEK"

merged_df = merged_df[[0, 1, 'AP', 'SEC']]
merged_df.columns = ['source', 'target', 'AP', 'SEC']

In [ ]:
merged_df_mod = merged_df[['source', 'target', 'AP']]
merged_df_mod.columns = ['source', 'target', 'type']
merged_df_mod1 = merged_df_mod[merged_df_mod['type'] != "None"]
merged_df_mod = merged_df[['source', 'target', 'SEC']]
merged_df_mod.columns = ['source', 'target', 'type']
merged_df_mod2 = merged_df_mod[merged_df_mod['type'] != "None"]
merged_df_mod = pd.concat([merged_df_mod1, merged_df_mod2])
merged_df_mod = merged_df_mod.drop_duplicates()
merged_df_mod.to_csv("./fig4_example.txt", sep="\t", header=True, index=False)

In [ ]:
# further visualize with cytoscape